# TikTok Shop — Data Pipeline Demo

**เริ่มใช้งาน:** เปิดจาก GitHub แล้วกดรันหัวข้อ 00 → 01 → 02 ได้เลย ไม่ต้องอัปโหลด ZIP หรือกรอก URL เอง

**โจทย์:** นำข้อมูลขายจำลองจาก Source ผ่าน Database และ Data Lake ไปสรุปยอดใน Warehouse

เราจะรันข้อมูลตั้งต้น (Run 1) แล้วเพิ่มข้อมูลใหม่ (Run 2) จากนั้นตรวจยอด ทดสอบข้อผิดพลาด และดูผลการทำงานค่ะ

> ข้อมูลทั้งหมดเป็นข้อมูลจำลอง ไม่ใช่ข้อมูลหรือค่าธรรมเนียมจริงของ TikTok Shop


## 00 — เตรียมโปรเจกต์

**ทำอะไร:** โหลด Code และไฟล์ข้อมูลสำหรับ Demo

**เห็นอะไร:** ตำแหน่งไฟล์ จำนวนข้อมูลตั้งต้น และพื้นที่เก็บผลลัพธ์


In [ ]:
#@title 00.1 เตรียมข้อมูลและพื้นที่ทำงาน
REPO_URL = "https://github.com/KankaweeK356/tiktok-shop-pipeline-demo" #@param {type:"string"}
RESET_DEMO = False #@param {type:"boolean"}

from pathlib import Path
import os, sys, subprocess
from IPython.display import display, Markdown

PROJECT_NAME = 'TikTok_Shop_Pipeline_Demo'
CWD = Path.cwd()
COLAB = Path('/content').exists()
DEFAULT_HOME = Path('/content') if COLAB else CWD

if (CWD/'pipeline'/'orchestrator.py').is_file():
    ROOT = CWD
else:
    target = DEFAULT_HOME/PROJECT_NAME
    if target.exists():
        # Re-running a demo can change local generated files. Avoid git pull:
        # it can fail on a dirty checkout and is unnecessary within one Runtime.
        print('พบไฟล์โปรเจกต์ใน Colab แล้ว — ใช้ Code เดิมโดยไม่ git pull ซ้ำ')
        print('หากต้องการ Code จาก GitHub เวอร์ชันล่าสุด ให้เริ่ม Runtime ใหม่')
    else:
        print('กำลังดาวน์โหลด Source และ Code จาก GitHub ...')
        subprocess.check_call(['git','clone','--depth','1',REPO_URL.strip(),str(target)])
    ROOT = target

# Check project completeness before trying to import. An update-only ZIP
# does not contain the main pipeline modules.
required_files = [
    'README.md', 'requirements.txt', 'pipeline/__init__.py',
    'pipeline/orchestrator.py', 'pipeline/generate.py',
    'pipeline/demo_tests.py', 'pipeline/quality.py',
    'pipeline/business.py', 'source/initial/Fact_Orders.csv',
]
missing_files = [p for p in required_files if not (ROOT/p).is_file()]
if missing_files:
    raise RuntimeError(
        f'โฟลเดอร์โปรเจกต์ยังไม่ครบ: {ROOT}\nขาดไฟล์: {missing_files}\n'
        'ตรวจ GitHub ว่าได้อัปโหลดชุด FULL ZIP ครบ ไม่ใช่เฉพาะ UPDATE ZIP'
    )

try:
    import duckdb, pyarrow
except ImportError:
    print('กำลังติดตั้งเครื่องมือสำหรับรัน Pipeline ...')
    subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements.txt')])

# Important: Google Colab/other notebooks can already have another package
# named `pipeline` loaded. Always resolve ours from this GitHub project.
import importlib
import importlib.util
project_location = str(ROOT.resolve())
sys.path = [p for p in sys.path if p != project_location]
sys.path.insert(0, project_location)
for module_name in list(sys.modules):
    if module_name == 'pipeline' or module_name.startswith('pipeline.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
pkg_init = ROOT/'pipeline'/'__init__.py'
pkg_spec = importlib.util.spec_from_file_location(
    'pipeline', str(pkg_init),
    submodule_search_locations=[str(pkg_init.parent.resolve())]
)
if pkg_spec is None or pkg_spec.loader is None:
    raise RuntimeError(f'ไม่สามารถโหลด Python package ของโปรเจกต์ได้: {pkg_init}')
project_pkg = importlib.util.module_from_spec(pkg_spec)
sys.modules['pipeline'] = project_pkg
pkg_spec.loader.exec_module(project_pkg)
print('Python package ที่ใช้งาน:', Path(project_pkg.__file__).resolve())
print('ชุดแก้ไข:', getattr(project_pkg, '__version__', 'unknown'))
from pipeline.orchestrator import run_batch,reset,summary
from pipeline.generate import generate
from pipeline.demo_tests import signature,error_suite,performance_suite
from pipeline.quality import verify,lineage
from pipeline.business import business_report
import pandas as pd
import matplotlib.pyplot as plt

WORKSPACE = ROOT/'output'/'main'
if RESET_DEMO:
    reset(WORKSPACE)
    print('เริ่มพื้นที่ Demo ใหม่แล้ว อย่าลืมเปลี่ยน RESET_DEMO เป็น False')
else:
    WORKSPACE.mkdir(parents=True,exist_ok=True)
if not (ROOT/'source'/'initial'/'Fact_Orders.csv').exists():
    generate(ROOT/'source',seed=7202)
print('โฟลเดอร์โปรเจกต์:', ROOT)
print('แบบฐานข้อมูล ER 1:',ROOT/'schemas/ER1_V3.dbml')
print('แบบคลังข้อมูล ER 2:',ROOT/'schemas/ER2_V5.dbml')
print('ไฟล์ผลลัพธ์:',WORKSPACE)
print('Order ตั้งต้น:',len(pd.read_csv(ROOT/'source/initial/Fact_Orders.csv')),'รายการ')


### ตรวจข้อมูลตั้งต้น

ดูว่าสินค้าและไฟล์ Source พร้อมก่อนรันหรือไม่


In [ ]:
display(pd.read_csv(ROOT/'source/initial/Dim_Product_Group.csv')[['Product_Group_Name','Base_Selling_Price']].head())
print('ไฟล์ข้อมูลตั้งต้น:', len(list((ROOT/'source'/'initial').glob('*.csv'))))
print('ไฟล์ข้อมูลรอบเพิ่ม:', len(list((ROOT/'source'/'incremental').glob('*.csv'))))


## 01 — Run 1: ข้อมูลตั้งต้น

**ทำอะไร:** โหลดข้อมูลเดือนตุลาคมผ่าน Source → SQLite → Parquet → DuckDB

**เห็นอะไร:** จำนวนรายการขาย ยอดขาย และผลตรวจข้อมูลรอบแรก


In [ ]:
#@title 01 รันข้อมูลตั้งต้น
run1 = run_batch(WORKSPACE, 'initial')
display(pd.DataFrame([{
    'ผลการรัน': 'สำเร็จ' if run1['status']=='SUCCESS' else run1['status'],
    'รายการขายที่ชำระเงิน': run1['paid_sales_rows'],
    'ยอดขายสุทธิ (บาท)': run1['net_revenue'],
    'ตรวจผ่าน (ข้อ)': run1['functional_passes'],
    'เวลารวม (วินาที)': run1['elapsed_seconds']
}]))
print('ไฟล์ผลลัพธ์:', WORKSPACE)


## 02 — Run 2: ข้อมูลใหม่

**ทำอะไร:** เพิ่ม Order ใหม่ อัปเดต Order ที่จ่ายเงินแล้ว และรับต้นทุนที่เข้าช้า

**เห็นอะไร:** ยอดก่อน/หลัง Run 2 และหลักฐานว่ากดรันซ้ำแล้วยอดไม่เพิ่มซ้ำ


In [ ]:
#@title 02 เพิ่มข้อมูลใหม่และตรวจการรันซ้ำ
before_signature = signature(WORKSPACE)
run2 = run_batch(WORKSPACE, 'incremental')
after_signature = signature(WORKSPACE)
replay = run_batch(WORKSPACE, 'incremental')
replay_signature = signature(WORKSPACE)
assert after_signature == replay_signature, 'รันซ้ำแล้วตัวเลขเปลี่ยน โปรดตรวจ Pipeline'

display(pd.DataFrame([
 {'ช่วงเวลา':'ก่อน Run 2', 'รายการที่จ่ายแล้ว':before_signature['orders'], 'ยอดขายสุทธิ (บาท)':before_signature['sales']},
 {'ช่วงเวลา':'หลัง Run 2', 'รายการที่จ่ายแล้ว':after_signature['orders'], 'ยอดขายสุทธิ (บาท)':after_signature['sales']},
 {'ช่วงเวลา':'รัน Run 2 ซ้ำ', 'รายการที่จ่ายแล้ว':replay_signature['orders'], 'ยอดขายสุทธิ (บาท)':replay_signature['sales']}
]))
print('รายการขายที่เปลี่ยน:',run2['metrics']['transform']['result']['changed_sales'])
print('ตรวจการรันซ้ำ: ผ่าน — ข้อมูลไม่เพิ่มซ้ำ')


## 03 — ตรวจยอดและที่มาของข้อมูล

**ทำอะไร:** เช็กยอดระหว่าง Database กับ Warehouse แล้วเลือก Order ตัวอย่างตามไปถึงรายงาน

**เห็นอะไร:** ผลผ่าน/ไม่ผ่าน และเส้นทางของ Order นั้น


In [ ]:
#@title 03 ตรวจยอดและที่มาของข้อมูล
latest = verify(WORKSPACE/'er1.sqlite', WORKSPACE/'er2.duckdb', WORKSPACE/'reports', 'final')
trace = lineage(WORKSPACE/'er1.sqlite', WORKSPACE/'er2.duckdb', 'final', WORKSPACE/'lake'/'control', WORKSPACE/'reports', limit=6)
checks = pd.read_csv(WORKSPACE/'reports'/'functional_final.csv')
print(f"ตรวจยอด: ผ่าน {latest['passed']}/{latest['checks']} ข้อ")
if latest['failed']:
    display(checks[checks['PASS']==False])
else:
    print('ไม่พบรายการตรวจสอบที่ไม่ผ่าน')
print('ตัวอย่างการตามข้อมูลจาก Source ถึงรายงาน:')
trace_preview = pd.read_csv(trace['file'])
display(trace_preview[['Source_Order_Item_ID','Source','ER1_Net_Revenue',
                       'ER2_Net_Revenue','Product_Group_ID','Monthly_KPI_Year_Month']].head(3))
print('รายละเอียดทั้งหมด:', WORKSPACE/'reports')


## 04 — ทดลอง Error และการกู้คืน

**ทำอะไร:** เลือกจุดที่ให้ระบบผิดพลาด แล้วแก้และรันต่อจากจุดเดิม

**เห็นอะไร:** จุดที่พัง สาเหตุ จุดที่เริ่มใหม่ และผลหลังแก้ไข


In [ ]:
#@title 04.1 เลือกกรณี Error
ERROR_MODE = 'specific' #@param ['specific','random','all']
SPECIFIC_CASE = 'T01' #@param ['E01','E02','T01','T02','L01','L02']
ERROR_SEED = 7202 #@param {type:"integer"}
import random
if ERROR_MODE=='specific':
    test_cases=[SPECIFIC_CASE]
elif ERROR_MODE=='random':
    test_cases=[random.Random(ERROR_SEED).choice(['E01','E02','T01','T02','L01'])]
else:
    test_cases=None
error_results = error_suite(ROOT/'output'/'test', cases=test_cases)
error_names={'E01':'SKU ไม่มีในข้อมูลสินค้า','E02':'หาไฟล์ Source ไม่พบ',
             'T01':'Voucher ผิดเงื่อนไข','T02':'Campaign ช่วงเวลาทับซ้อน',
             'L01':'โหลด Warehouse ล้มเหลว','L02':'ส่งข้อมูลเดิมซ้ำ'}
error_show = error_results[['case','stage','detected','retry_from','passed']].copy()
error_show.insert(1,'กรณีทดสอบ',error_show['case'].map(error_names))
error_show.rename(columns={'case':'รหัส','stage':'ขั้นที่ทดสอบ','detected':'ตรวจพบ',
                           'retry_from':'เริ่มใหม่จาก','passed':'ผ่าน'}, inplace=True)
display(error_show)
assert error_results['passed'].all(), 'มีกรณีทดสอบที่ยังไม่ผ่าน'
print('ผลทดสอบ:',int(error_results['passed'].sum()),'/',len(error_results),'ผ่าน')


### ทดสอบ Error ครบทุกกรณี (ใช้เก็บหลักฐาน)

ระบบจะทดสอบครบ 6 กรณีและสรุปว่าผ่านกี่กรณี


In [ ]:
#@title 04.2 ทดสอบ Error ครบทุกกรณี
RUN_ALL_SIX_ERROR_CASES = True #@param {type:"boolean"}
if RUN_ALL_SIX_ERROR_CASES:
    six = error_suite(ROOT/'output'/'test', cases=None)
    print('ผลทดสอบ Error ทั้งหมด:',int(six['passed'].sum()),'/',len(six),'ผ่าน')
    assert six['passed'].all(), 'ยังมีกรณี Error ที่ไม่ผ่าน'
    print('รายงาน:',ROOT/'output'/'test'/'error_suite_summary.csv')
else:
    print('ข้ามการทดสอบครบ 6 กรณี')


## 05 — ประสิทธิภาพและผลธุรกิจ

**ทำอะไร:** จับเวลารันข้อมูลหลายขนาด แล้วเปิดรายงานยอดขายและกำไร

**เห็นอะไร:** เวลาประมวลผล จำนวนรายการต่อวินาที และกำไรที่เหลือในแต่ละเดือน


In [ ]:
#@title 05.1 วัดเวลาประมวลผล
RUN_PERFORMANCE_TEST = True #@param {type:"boolean"}
if RUN_PERFORMANCE_TEST:
    perf = performance_suite(ROOT/'output'/'performance', sizes=(100,300,800))
    display(perf[['initial_source_orders','latency_seconds','throughput_paid_items_per_second']]
            .rename(columns={'initial_source_orders':'จำนวน Order ตั้งต้น',
                             'latency_seconds':'เวลารวม (วินาที)',
                             'throughput_paid_items_per_second':'รายการขายต่อวินาที'}))
    fig,ax=plt.subplots(figsize=(7,3.5))
    ax.plot(perf['initial_source_orders'], perf['latency_seconds'], marker='o')
    ax.set(title='Processing time by order volume', xlabel='Number of orders',ylabel='Seconds')
    ax.grid(alpha=.2)
    plt.tight_layout();plt.show()
    print('ผลวัดจากเครื่องที่กำลังรัน ผลจริงอาจต่างกันในแต่ละครั้ง')
else:
    print('ข้ามการวัดประสิทธิภาพ')


In [ ]:
#@title 05.2 ดูยอดขายและกำไร
monthly, groups = business_report(WORKSPACE/'er2.duckdb', WORKSPACE/'reports')
print('ยอดขายและกำไรรายเดือน:')
display(monthly[['Year_Month','Net_Revenue','Contribution_Profit',
                 'Profit_Margin_Pct','Profit_Protection_Headroom','Profit_Protection_Status']]
        .rename(columns={'Year_Month':'เดือน','Net_Revenue':'ยอดขายสุทธิ',
                         'Contribution_Profit':'กำไรหลังต้นทุน','Profit_Margin_Pct':'อัตรากำไร (%)',
                         'Profit_Protection_Headroom':'กำไรเกิน/ขาดเกณฑ์',
                         'Profit_Protection_Status':'เกณฑ์กำไรขั้นต่ำ'}))
print('เพดานคอมมิชชันแยกกลุ่มสินค้า (สมมติยอดขายและต้นทุนอื่นคงเดิม):')
group_view = groups[['Product_Group_Name','Net_Revenue','Max_Commission_Rate_If_All_Else_Fixed']].copy()
group_view['Max_Commission_Rate_If_All_Else_Fixed']=(group_view['Max_Commission_Rate_If_All_Else_Fixed']*100).round(2)
display(group_view.rename(columns={'Product_Group_Name':'กลุ่มสินค้า','Net_Revenue':'ยอดขายสุทธิ',
                                  'Max_Commission_Rate_If_All_Else_Fixed':'เพดานคอมมิชชัน (%)'}))
fig,ax=plt.subplots(figsize=(7,3.5))
ax.bar(monthly['Year_Month'], monthly['Net_Revenue'], label='Net revenue')
ax.plot(monthly['Year_Month'], monthly['Contribution_Profit'], marker='o',label='Profit')
ax.set(title='Monthly revenue and profit (mock data)', xlabel='Month', ylabel='Baht')
ax.legend();plt.tight_layout();plt.show()
print('รายงาน:',WORKSPACE/'reports')


## ไฟล์ผลลัพธ์อยู่ที่ไหน?

- **SQLite (`output/main/er1.sqlite`)** — ข้อมูลธุรกิจใน ER 1
- **Parquet (`output/main/lake/`)** — ข้อมูลแต่ละขั้นของ Data Lake
- **DuckDB (`output/main/er2.duckdb`)** — ข้อมูลพร้อมวิเคราะห์ใน ER 2
- **Reports (`output/main/reports/`)** — ผลตรวจยอด ที่มาของข้อมูล และ KPI
- **Test (`output/test/`)** — ผลการทดลอง Error ไม่กระทบข้อมูลหลัก

**เริ่มใหม่:** ตั้ง `RESET_DEMO=True` ในขั้น 00 แล้วรันหนึ่งครั้ง จากนั้นปรับกลับเป็น `False` ก่อน Run 1

> ค่าต้นทุนและค่าธรรมเนียมเป็นสมมติฐานของโจทย์นี้ ผล Affiliate Scenario เป็นเพดานตามยอดขายและต้นทุนที่กำหนด ไม่ใช่การคาดการณ์ยอดขายที่จะเพิ่มขึ้น


### 05.3 — เตรียมรายงานสำหรับ GitHub

**ทำอะไร:** รวมผล Run 1, Run 2 และ Test ที่ผ่านครบเป็น ZIP สำหรับอัปโหลดกลับ GitHub

**เห็นอะไร:** รายชื่อไฟล์รายงานและลิงก์ดาวน์โหลด โดย **ไม่ส่งข้อมูลกลับ GitHub อัตโนมัติ**

In [ ]:
#@title 05.3 สร้างรายงานสำหรับ GitHub
PREPARE_GITHUB_REPORTS = True #@param {type:"boolean"}
DOWNLOAD_GITHUB_REPORTS_ZIP = False #@param {type:"boolean"}
from pipeline.publication import export_github_reports, EvidenceNotReady
if PREPARE_GITHUB_REPORTS:
    try:
        published = export_github_reports(ROOT)
        print('เตรียมรายงานที่ตรวจแล้วเรียบร้อย:', published['zip_path'])
        print('ให้นำโฟลเดอร์ reports/ ใน ZIP ขึ้น GitHub ด้วย Upload files และ Commit changes')
        if DOWNLOAD_GITHUB_REPORTS_ZIP:
            from google.colab import files
            files.download(published['zip_path'])
    except EvidenceNotReady as exc:
        print('ยังไม่สร้างรายงาน GitHub:', exc)
else:
    print('ข้ามการสร้างชุดรายงาน GitHub')

### 05.4 — บันทึกหรือเคลียร์ผลบน Google Drive (ไม่บังคับ)

**ทำอะไร:** เลือกเก็บ Snapshot ของ Database, Lake และรายงานลง **Drive ของผู้รันเอง** หรือเคลียร์เฉพาะ Snapshot ของ Demo

**เห็นอะไร:** ตำแหน่งที่บันทึก หรือจำนวน Snapshot ที่ถูกลบ ข้อมูล GitHub ไม่เปลี่ยน

In [ ]:
#@title 05.4 จัดการผลลัพธ์บน Google Drive (ไม่บังคับ)
DRIVE_ACTION = 'none' #@param ['none','save','clear']
CLEAR_CONFIRM = '' #@param {type:"string"}
from pipeline.drive_storage import save_to_drive, clear_drive_outputs
if DRIVE_ACTION == 'none':
    print('ไม่ได้เชื่อม Drive — ใช้ผลลัพธ์ใน Colab ตามปกติ')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_home = Path('/content/drive/MyDrive')
    if DRIVE_ACTION == 'save':
        saved = save_to_drive(ROOT, drive_home)
        print('บันทึกข้อมูล Demo ลง Drive แล้ว:', saved['path'])
    elif DRIVE_ACTION == 'clear':
        removed = clear_drive_outputs(drive_home, CLEAR_CONFIRM)
        print('เคลียร์ Snapshot ของ Demo:', removed['cleared_snapshots'], 'ชุด')
        print('ไม่ลบโฟลเดอร์อื่นและไม่แก้ไฟล์ใน GitHub')